# Data augmentation for nighttime detection PAPI dataset

## Lessons learned — current augmentation strategy (updated 2026-06-18)

The **offline 3× pre-generation** below is kept for reference but is **superseded by online
augmentation during training** (`workflows/scripts/train_detector_model.py --weather-aug` plus
`--mosaic/--erasing/--degrees/...`), which augments per-epoch with no disk blow-up. Findings
from the weather-robust nano runs:

**Colour IS the class (red vs white lamp) — colour aug must preserve hue.**
- `hsv_h = hsv_s = 0` **always**: hue/saturation jitter relabels red↔white (the retired serving
  model's `hsv_s=0.7` is the cautionary tale). Only `hsv_v` (brightness) is colour-safe.
- For season / time-of-day variety use a **colour-TEMPERATURE shift** (warm/cool white-balance —
  a simplified *Planckian jitter*): it tints the scene without crossing red↔white. See the
  `warm`/`cool` conditions in `workflows/scripts/weather_aug.py`.

**Use OpenCV, NOT albumentations, for weather/photometric aug.**
- AlbumentationsX's `albucore` SIMD backend **corrupts asynchronous CUDA training** on this stack
  (torch 2.5.1+cu121 / RTX 4070): every weather-augmented run crashed with assorted CUDA faults,
  while Ultralytics' OpenCV mosaic ran fine. `weather_aug.py` is therefore **pure OpenCV/NumPy**
  (also AGPL-free). `albumentations` is uninstalled — the original cells would not run as-is.

**What helps vs hurts (small, colour-coded objects = PAPI lamps).**
- ✅ **mosaic** (`mosaic=1.0`, `close_mosaic=10`) — strongest regulariser, great for small objects.
- ✅ mild **rotation** (`degrees≈10`, drone roll), random **erasing** (`≈0.4`), `scale=0.5`,
  weather + warm/cool tints.
- ❌ **perspective / shear** — obscure small objects (Ultralytics guidance) → dropped.
- ❌ **mixup / cutmix** — blur small objects and blend colours.

**Training-pipeline gotchas from these runs.**
- **AMP (fp16) overflows under heavy aug** (mosaic composites + bright weather) → NaN / "illegal
  instruction". Train heavy-aug runs in **FP32 (`--no-amp`)**.
- **`multi_scale`** enlarges the canvas to ~1.5× → VRAM spill at FP32; `scale=0.5` gives scale
  variety inside the fixed canvas, so prefer it and skip `multi_scale`.
- The OpenCV weather transform is **worker-safe** (`weather_aug.WeatherAug` pickles to dataloader
  workers) so training runs `workers>0`.
- **Overfitting is diagnosed by the train/val loss *gap*, not the absolute loss.** Augmentation
  narrows the gap and delays overfitting, but with ~11 flights of near-duplicate frames val mAP
  plateaus regardless — the real cure is more flights/airports (data, not augmentation).

## Imports

In [ ]:
import os
import sys

import cv2
import numpy as np

# Colour-safe weather aug is now pure OpenCV (NOT albumentations — its albucore backend breaks
# async CUDA training; see the lessons-learned cell above). Transforms live in weather_aug.py.
sys.path.insert(0, os.path.join("..", "scripts"))
from weather_aug import apply_weather  # noqa: E402

> **Augmentation safety note.** This pipeline augments **train only** — augmenting val/test
> makes eval metrics optimistic, so val/test images are copied **verbatim**. The effects here are
> pure-OpenCV and **non-spatial** (weather veils + warm/cool colour-temperature tints), so bounding
> boxes are unchanged and labels are reused verbatim. Hue/saturation are never jittered (colour is
> the class); horizontal flip is not applied (it reverses lamp order). For real training prefer the
> trainer's **online** aug (`train_detector_model.py --weather-aug ...`), which adds mosaic +
> geometric variety per-epoch — see the lessons-learned cell at the top.

> **Augmentation safety note.** This pipeline must augment **train only** — augmented
val/test would make eval metrics optimistic; copy val/test images **verbatim** instead.
`HorizontalFlip` is removed because it reverses left-to-right lamp order and corrupts the
ordered glideslope labels; brightness/contrast are bounded to avoid pushing dim-red toward white.


In [ ]:
# =========================================================
# OFFLINE 3x augmentation (reference — prefer ONLINE aug in train_detector_model.py).
# Pure-OpenCV, colour-safe, NON-SPATIAL weather/lighting → bounding boxes unchanged.
# =========================================================

INPUT_DATASET = "PAPI_Night.yolo26"
OUTPUT_DATASET = "PAPI_Night_Augmented.yolo26"

SPLITS = ["train", "valid", "test"]
SPLITS_TO_AUGMENT = {"train"}  # augment TRAIN ONLY (augmenting val/test => optimistic eval)

# Colour-safe conditions from weather_aug.py (non-spatial: labels are reused verbatim).
WEATHER = ["rain", "fog", "haze", "snow", "warm", "cool"]
rng = np.random.default_rng(0)  # deterministic

for split in SPLITS:
    in_img = os.path.join(INPUT_DATASET, split, "images")
    in_lbl = os.path.join(INPUT_DATASET, split, "labels")
    out_img = os.path.join(OUTPUT_DATASET, split, "images")
    out_lbl = os.path.join(OUTPUT_DATASET, split, "labels")
    os.makedirs(out_img, exist_ok=True)
    os.makedirs(out_lbl, exist_ok=True)

    augment = split in SPLITS_TO_AUGMENT
    print(f"\nProcessing split: {split} (augment={augment})")

    for filename in os.listdir(in_img):
        if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        stem = filename.rsplit(".", 1)[0]

        image = cv2.imread(os.path.join(in_img, filename))  # BGR
        if image is None:
            print(f"Could not read image: {filename}")
            continue

        label_path = os.path.join(in_lbl, stem + ".txt")
        labels_txt = open(label_path).read() if os.path.exists(label_path) else ""

        # Always copy the original (image + labels) verbatim.
        cv2.imwrite(os.path.join(out_img, filename), image)
        with open(os.path.join(out_lbl, stem + ".txt"), "w") as f:
            f.write(labels_txt)

        if not augment:
            continue

        # Weather is NON-SPATIAL, so the SAME labels apply to every augmented copy.
        for i in range(2):  # original + 2 augs => 3x on train
            cond = str(rng.choice(WEATHER))
            aug = apply_weather(image, cond, severity="medium", rng=rng)
            cv2.imwrite(os.path.join(out_img, f"{stem}_aug{i}.jpg"), aug)
            with open(os.path.join(out_lbl, f"{stem}_aug{i}.txt"), "w") as f:
                f.write(labels_txt)

    print(f"Finished split: {split}")

print("\nDataset augmentation complete (colour-safe OpenCV weather/tints).")